In [65]:
import os
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, pairwise_distances, silhouette_score

In [66]:
embeddings_dir = Path('.')
filtered_embeddings_dir = embeddings_dir / 'filtered_embeddings'
results_dir = embeddings_dir / 'results' / 'cluster_analysis'
results_dir.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(embeddings_dir / 'filtered_poems.csv', engine='python')
embedding_files = sorted(filtered_embeddings_dir.glob('*.npy'))

embeddings = {}
for path in embedding_files:
    arr = np.load(path).astype(np.float64)
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    embeddings[path.stem] = arr / norms

print(f'Poems: {len(data)}')
print(f'Embeddings: {len(embeddings)}')
print(sorted(embeddings.keys()))

Poems: 2853
Embeddings: 11
['baseline', 'd2v_line_d100_w5_dbow', 'd2v_line_d100_w5_dm', 'd2v_poem_d100_w5_dbow', 'd2v_poem_d100_w5_dm', 'w2v_d100_w5_cbow_avg', 'w2v_d100_w5_cbow_idf', 'w2v_d100_w5_cbow_sif', 'w2v_d100_w5_sg_avg', 'w2v_d100_w5_sg_idf', 'w2v_d100_w5_sg_sif']


# Optimal K-value search by silhouette

In [68]:
# Parameters
K_MIN = 3
K_MAX = 30
SILHOUETTE_SAMPLE = 1500     # maximum sample size for silhouette computation
PAIRWISE_SAMPLE = 5000       # sample size for pairwise similarity analysis
RANDOM_STATE = 42


In [69]:
from sklearn.utils import check_random_state

def safe_silhouette(matrix, cluster_labels):
    unique = np.unique(cluster_labels)
    if len(unique) < 2 or len(unique) >= len(cluster_labels):
        return np.nan
    return silhouette_score(
        matrix,
        cluster_labels,
        metric='cosine',
        sample_size=min(1500, len(cluster_labels)),
        random_state=42,
    )

def k_search_on_matrix(matrix, k_min=K_MIN, k_max=K_MAX, random_state=RANDOM_STATE):
    """Compute inertia (SSE) and silhouette for k in [k_min, k_max].

    Returns a dict with keys 'ks', 'sse' and 'silhouette'.
    """
    n_samples = matrix.shape[0]
    ks = list(range(k_min, min(k_max, n_samples - 1) + 1))
    sse = []
    sils = []
    for k in ks:
        try:
            km = KMeans(n_clusters=k, random_state=random_state, n_init=20)
            labels = km.fit_predict(matrix)
            sse.append(float(km.inertia_))
            sils.append(safe_silhouette(matrix, labels))
        except Exception as exc:
            print(f'k={k} failed: {exc}')
            sse.append(np.nan)
            sils.append(np.nan)
    return {'ks': ks, 'sse': sse, 'silhouette': sils}

def plot_k_search(results, title, out_prefix):
    """Save elbow (inertia) and silhouette plots for a given k-search result."""
    ks = results['ks']
    sse = results['sse']
    sils = results['silhouette']

    plt.figure(figsize=(8, 3.5))
    plt.plot(ks, sse, marker='o')
    plt.xlabel('k')
    plt.ylabel('Inertia (SSE)')
    plt.title(f'{title} — Elbow')
    plt.grid(True)
    elbow_path = f'figures/k_search/{out_prefix}_elbow.png'
    plt.savefig(elbow_path, dpi=150, bbox_inches='tight')
    plt.close()

    plt.figure(figsize=(8, 3.5))
    plt.plot(ks, sils, marker='o')
    plt.xlabel('k')
    plt.ylabel('Silhouette')
    plt.title(f'{title} — Silhouette')
    plt.grid(True)
    sil_path = f'figures/k_search/{out_prefix}_silhouette.png'
    plt.savefig(sil_path, dpi=150, bbox_inches='tight')
    plt.close()

def pick_k_by_silhouette(results):
    """Return k with maximum silhouette score; None if silhouettes are all NaN."""
    sils = np.array(results['silhouette'], dtype=np.float64)
    ks = np.array(results['ks'], dtype=int)
    if np.all(np.isnan(sils)):
        return None
    valid = ~np.isnan(sils)
    best_idx = np.argmax(sils[valid])
    return int(ks[valid][best_idx])


In [70]:
EMBEDDING_DICT = {
    'baseline' : 'baseline',
    'w2v_d100_w5_cbow_avg' : 'word_cbow_avg',
    'w2v_d100_w5_cbow_idf' : 'word_cbow_idf',
    'w2v_d100_w5_cbow_sif' : 'word_cbow_sif',
    'w2v_d100_w5_sg_avg' : 'word_sg_avg',
    'w2v_d100_w5_sg_idf' : 'word_sg_idf',
    'w2v_d100_w5_sg_sif' : 'word_sg_sif',
    'd2v_line_d100_w5_dm': 'line_dm',
    'd2v_line_d100_w5_dbow': 'line_dbow',
    'd2v_poem_d100_w5_dm': 'poem_dm',
    'd2v_poem_d100_w5_dbow': 'poem_dbow'
    
}

In [71]:
k_by_embedding = {}
for embedding_name, matrix in embeddings.items():
    print(EMBEDDING_DICT[embedding_name])

    k_dict = k_search_on_matrix(matrix) 
    plot_k_search(k_dict, EMBEDDING_DICT[embedding_name], EMBEDDING_DICT[embedding_name])
    k = pick_k_by_silhouette(k_dict)
    k_by_embedding[EMBEDDING_DICT[embedding_name]] = k

baseline
line_dbow
line_dm
poem_dbow
poem_dm
word_cbow_avg
word_cbow_idf
word_cbow_sif
word_sg_avg
word_sg_idf
word_sg_sif


In [72]:
k_by_embedding

{'baseline': 10,
 'line_dbow': 3,
 'line_dm': 4,
 'poem_dbow': 3,
 'poem_dm': 3,
 'word_cbow_avg': 3,
 'word_cbow_idf': 3,
 'word_cbow_sif': 3,
 'word_sg_avg': 3,
 'word_sg_idf': 3,
 'word_sg_sif': 3}

# Run clustering

In [73]:
top_20_poets = data['Poet'].value_counts().head(20).index

tasks = {
    'Emotion': {
        'mask': np.ones(len(data), dtype=bool),
        'labels': data['Emotion'].astype(str).to_numpy(),
        'n_clusters': data['Emotion'].nunique(),
    },
    'Period': {
        'mask': np.ones(len(data), dtype=bool),
        'labels': data['Period'].astype(str).to_numpy(),
        'n_clusters': data['Period'].nunique(),
    },
    'PoetTop20': {
        'mask': data['Poet'].isin(top_20_poets).to_numpy(),
        'labels': data.loc[data['Poet'].isin(top_20_poets), 'Poet'].astype(str).to_numpy(),
        'n_clusters': 20,
    },
}

task_summary_rows = []
for task_name, task in tasks.items():
    labels = pd.Series(task['labels'])
    task_summary_rows.append({
        'task': task_name,
        'n_samples': len(labels),
        'n_clusters': task['n_clusters'],
        'largest_class_size': int(labels.value_counts().max()),
        'smallest_class_size': int(labels.value_counts().min()),
    })

task_summary = pd.DataFrame(task_summary_rows)
task_summary

,task,n_samples,n_clusters,largest_class_size,smallest_class_size
0,Emotion,2853,6,820,210
1,Period,2853,6,894,19
2,PoetTop20,536,20,51,21


In [74]:
TOP_LEVEL_TAGS = [
    'Living',
    'Love',
    'Nature',
    'Activities',
    'Arts & Sciences',
    'Relationships',
    'Social Commentaries',
    'Religion',
    'The Body',
    'Mythology & Folklore',
]

def make_agglomerative(n_clusters):
    try:
        return AgglomerativeClustering(n_clusters=n_clusters, linkage='average', metric='cosine')
    except TypeError:
        return AgglomerativeClustering(n_clusters=n_clusters, linkage='average', affinity='cosine')

def safe_silhouette(matrix, cluster_labels):
    unique = np.unique(cluster_labels)
    if len(unique) < 2 or len(unique) >= len(cluster_labels):
        return np.nan
    return silhouette_score(
        matrix,
        cluster_labels,
        metric='cosine',
        sample_size=min(1500, len(cluster_labels)),
        random_state=42,
    )

In [75]:
run_rows = []

for embedding_name, matrix in embeddings.items():
    for task_name, task in tasks.items():
        task_matrix = matrix[task['mask']]
        labels = task['labels']
        n_clusters = k_by_embedding[EMBEDDING_DICT[embedding_name]]

        for seed in range(10):
            cluster_labels = KMeans(n_clusters=n_clusters, random_state=seed, n_init=20).fit_predict(task_matrix)
            run_rows.append({
                'embedding': embedding_name,
                'task': task_name,
                'algorithm': 'kmeans',
                'run': seed,
                'n_samples': len(labels),
                'n_clusters': n_clusters,
                'nmi': normalized_mutual_info_score(labels, cluster_labels),
                'ari': adjusted_rand_score(labels, cluster_labels),
                'silhouette': safe_silhouette(task_matrix, cluster_labels),
            })

        cluster_labels = make_agglomerative(n_clusters).fit_predict(task_matrix)
        run_rows.append({
            'embedding': embedding_name,
            'task': task_name,
            'algorithm': 'agglomerative',
            'run': 0,
            'n_samples': len(labels),
            'n_clusters': n_clusters,
            'nmi': normalized_mutual_info_score(labels, cluster_labels),
            'ari': adjusted_rand_score(labels, cluster_labels),
            'silhouette': safe_silhouette(task_matrix, cluster_labels),
        })

run_metrics = pd.DataFrame(run_rows)

summary = (
    run_metrics.groupby(['embedding', 'task', 'algorithm'], as_index=False)
    .agg(
        n_samples=('n_samples', 'first'),
        n_clusters=('n_clusters', 'first'),
        nmi_mean=('nmi', 'mean'),
        nmi_std=('nmi', 'std'),
        ari_mean=('ari', 'mean'),
        ari_std=('ari', 'std'),
        silhouette_mean=('silhouette', 'mean'),
        silhouette_std=('silhouette', 'std'),
    )
    .fillna(0.0)
    .sort_values(['task', 'algorithm', 'nmi_mean'], ascending=[True, True, False])
    .reset_index(drop=True)
)

run_metrics.to_csv(results_dir / 'clustering_run_metrics.csv', index=False)
summary.to_csv(results_dir / 'clustering_summary.csv', index=False)
task_summary.to_csv(results_dir / 'task_summary.csv', index=False)

summary

,embedding,task,algorithm,n_samples,n_clusters,nmi_mean,nmi_std,ari_mean,ari_std,silhouette_mean,silhouette_std
0,baseline,Emotion,agglomerative,2853,10,0.010607,0.000000,-0.000109,0.000000,0.045572,0.000000
1,w2v_d100_w5_cbow_idf,Emotion,agglomerative,2853,3,0.005754,0.000000,0.001583,0.000000,0.382050,0.000000
2,d2v_line_d100_w5_dm,Emotion,agglomerative,2853,4,0.003948,0.000000,0.000191,0.000000,0.239144,0.000000
3,w2v_d100_w5_cbow_sif,Emotion,agglomerative,2853,3,0.003907,0.000000,0.000713,0.000000,0.525889,0.000000
4,w2v_d100_w5_cbow_avg,Emotion,agglomerative,2853,3,0.002920,0.000000,0.000488,0.000000,0.665694,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
61,d2v_line_d100_w5_dbow,PoetTop20,kmeans,536,3,0.213768,0.006971,0.063645,0.002499,0.190221,0.001307
62,w2v_d100_w5_cbow_idf,PoetTop20,kmeans,536,3,0.210131,0.001882,0.063115,0.000291,0.265035,0.002131
63,baseline,PoetTop20,kmeans,536,10,0.201101,0.010967,0.054460,0.006492,0.080296,0.020227
64,w2v_d100_w5_cbow_sif,PoetTop20,kmeans,536,3,0.152158,0.002479,0.045216,0.000725,0.131880,0.000826
